This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide

## Setting Entry

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
class SimType(Enum):
    TWOFOURFIVE = 1
    # AMBE = 2
    DDFUSION = 2

sim_type = helpers.get_input_required(
    """\
Which test data should be used?
1: 2.45 MeV neutron sim
2: D-D fusion sim
""",
    [SimType.TWOFOURFIVE, SimType.DDFUSION],
    lambda x: SimType(int(x))
)

In [ ]:
norm_input = helpers.get_input_with_default(
    "Normalize N based on R? [y/N] >", "n", str
)
should_normalize = norm_input.lower() == "y"

In [ ]:
phi0_input = helpers.get_input_with_default(
    "Create phi0 based on expected output? [y/N] >", "n", str
)
make_phi0 = phi0_input.lower() == "y"

In [ ]:
# ambe_scaling = 1
# if sim_type == SimType.AMBE:
#     ambe_scaling = helpers.get_input_with_default(
#         "Enter scaling factor for AmBe ISO spectrum, or press Enter for default (0.1) >", 
#         0.1,
#         float
#     )

## Data Processing

### Loading

In [ ]:
R = load_neutron_response_matrix(
    # Path("response_matrix_geant_point"),
    # Path("response_matrix_geant_gaussian"),
    # Path("response_matrix_geant_revised"),
    # Path("response_matrix_geant_revised_20keV_res"),
    # Path("response_matrix_geant_revised_50keV_res"),
    # Path("response_matrix_geant_Tbird_WithSig"),
    # Path("response_matrix_50keV_sigma"),
    # Path("response_matrix_50keV_sigma_NoFWHM"),
    Path("response_matrix_R3"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
# Experiment: Use log values
# with np.errstate(invalid="ignore", divide="ignore"):
#     log_counts = np.log(R.counts)
#     log_counts = np.nan_to_num(log_counts, posinf=np.nan, neginf=np.nan)
#     R = NDHistogram(log_counts, R.midpoints)

In [ ]:
R.counts.shape

In [ ]:
np.nansum(R.counts, axis=0)

In [ ]:
# widths = [15, 15, 20, 15, 15, 15, 15, 15]

base_path = Path("unfolding_test")
if sim_type == SimType.TWOFOURFIVE:
    # filename = "sim_2.450_MeV.csv.npy"
    # filename = "sim_2.450_MeV_Tbird_NoRes_Bin1000.csv.npy"
    filename = "neutron_2.45_MeV_LightOutput.csv.npy"
    test_file = base_path / filename
# elif sim_type == SimType.AMBE:
#     test_file = base_path / "output-AmBe.txt"
elif sim_type == SimType.DDFUSION:
    # filename = "sim_gaussian_2.450_MeV.csv.npy"
    # filename = "sim_2.450_MeV_Tbird_WithRes_Bin1000.csv.npy"
    filename = "neutron_DD_LightOutput.csv.npy"
    test_file = base_path / filename

# if sim_type in [SimType.TWOFOURFIVE, SimType.AMBE]:
#     df = pd.read_fwf(test_file, widths=widths)
# else:
#     df = pd.read_csv(test_file, sep="\t")
L_array = np.load(test_file)

bins = np.arange(bins_min, bins_max + bins_width, bins_width)
# cut = pd.cut(df["det_pulse (MeVee)"], bins.tolist())
# cut_index = cut.cat.categories
# new_df = pd.DataFrame(
#     df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)
# )
np_cps, *_ = np.histogram(L_array, bins=bins)

# old_index = new_df.index
# if not isinstance(old_index, pd.IntervalIndex):
#     raise RuntimeError(
#         f"DataFrame created from cut/groupby for {test_file.name} "
#         + "was not an IntervalIndex as expected"
#     )
# mids = old_index.mid.to_series(index=cut_index)

# np_cps = new_df["NPS"].to_numpy().reshape(-1, 1)
# np_Ls = mids.to_numpy()
np_cps = np_cps.reshape(-1, 1)
np_Ls = (bins[1:] + bins[:-1]) / 2

N = NDHistogram(np_cps, [np_Ls, np.ones(1)])

In [ ]:
N.shape

In [ ]:
# # Experiment: use log counts
# with np.errstate(invalid="ignore", divide="ignore"):
#     log_counts = np.log(N.counts)
#     log_counts = np.nan_to_num(log_counts, posinf=np.nan, neginf=np.nan)
#     N = NDHistogram(log_counts, N.midpoints)

In [ ]:
# ambe_file = Path("unfolding_test") / "ambe_spectrum.csv"
# ambe_df = pd.read_fwf(ambe_file)
# # ambe_df

In [ ]:
e_uncerts = [(0.75, None, 5.0625),
 (1.0, None, 1.0),
 (1.25, 0.6875, 0.875),
 (1.5, 0.5625, 0.8125),
 (1.75, 0.5, 0.75),
 (2.0, 0.5, 0.8125),
 (2.25, 0.4375, 0.8125),
 (2.5, 0.375, 0.3125),
 (2.75, 0.375, 0.3125),
 (3.0, 0.3125, 0.3125),
 (3.25, 0.3125, 0.3125),
 (3.5, 0.3125, 0.3125),
 (3.75, 0.3125, 0.3125),
 (4.0, 0.375, 0.375),
 (4.25, 0.4375, 0.3125),
 (4.5, 0.1875, 1.25),
 (4.75, 0.375, 1.1875),
 (5.0, 0.625, 0.3125),
 (5.25, 0.5625, 0.5),
 (5.5, 0.875, 0.3125)]

### Processing

In [ ]:
# ambe_limited = ambe_df.query("si1 <= 6")
# ambe_E = ambe_limited["si1"]
# ambe_N = ambe_limited["sp1"]

In [ ]:
if should_normalize:
    R_max = R.counts.max(axis=0, keepdims=True)
    N_max = N.counts.max()
    norm_factor = _nan_divide(N_max, R_max)
    N_norm = N.counts * norm_factor
    N = NDHistogram(N_norm, R.midpoints)

In [ ]:
if make_phi0:
    R_L_mids, R_E_mids = R.midpoints
    reduced_L_mids = np.array([R_L_mids.mean()])
    if sim_type == SimType.TWOFOURFIVE:  # 2.45 MeV
        phi0_counts = np.ones_like(R_E_mids)
        phi0_counts = phi0_counts.reshape(1, -1)
        peak_idx = np.argmax(R_E_mids >= 2.45)
        phi0_counts[0, peak_idx] = 100
    elif sim_type == SimType.AMBE:  # AmBe
        phi0_counts = np.interp(R_E_mids, ambe_E, ambe_N)
        phi0_counts = phi0_counts.reshape(1, -1)
    elif sim_type == SimType.DDFUSION:  # D-D fusion
        phi0_counts = None

    if phi0_counts is None:
        phi0 = None
    else:
        phi0 = NDHistogram(phi0_counts, [reduced_L_mids, R_E_mids])
else:
    phi0 = None

In [ ]:
# print(R.counts.shape)
# sigma = NDHistogram(np.sqrt(N.counts), N.midpoints)
# new_R, new_N, new_phi, new_sigma = clean_data(R, N, NDHistogram(
#     np.ones((1, R.shape[1])),
#     [np.ones(1), R.midpoints[1]]
# ), sigma)
# print(new_R.shape)
# print(new_N.shape)
# print(new_phi.shape)
# print(new_sigma.shape)

In [ ]:
np.isclose(R.midpoints[0], N.midpoints[0]).all()

In [ ]:
(R.midpoints[0] == N.midpoints[0])

In [ ]:
phi, unfold_info = unfold_spectrum(
    R,
    N,
    L_cut=0.05,
    # tolerance=1e-6,
    max_iterations=1000,
    full_info=True,
    phi0=phi0
)
chis = unfold_info["chis"]
errors = unfold_info["errors"]

In [ ]:
len(errors)

In [ ]:
phi_flat = phi.counts.reshape(-1)
phi_mids = phi.midpoints[1]

In [ ]:
phi_flat.sum()

### Peak Finding

In [ ]:
peaks, p_data = find_peaks(
    phi_flat,
    prominence=np.nanmax(phi_flat) / 100
)
print(peaks)
if len(peaks) > 0:
    print(phi_mids[peaks])
    print(phi_flat[peaks])

## Plotting

### Standard

In [ ]:
figsize = (9, 6)
fig, ax = plt.subplots(figsize=figsize)
ax.plot(range(len(chis)), chis)
ax.set(yscale="log", ylabel="Chi^2/n", xlabel="Iteration", title="Stopping Criteria")
plt.show()

In [ ]:
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)

if sim_type == SimType.DDFUSION:
    peak_lo, peak_hi = 2, 3
elif sim_type == SimType.TWOFOURFIVE:
    peak_lo, peak_hi = 2.2, 2.6
else:
    peak_lo, peak_hi = None, None
if peak_lo is not None:
    xerr = []
    for mid in phi_mids:
        matches = [
            (lo, hi) for E, lo, hi in e_uncerts
            if peak_lo <= E <= peak_hi and np.isclose(mid, E)
        ]
        if len(matches) == 0:
            xerr.append((np.nan, np.nan))
        else:
            xerr.append(matches[0])
    xerr = list(zip(*xerr))
else:
    xerr = None

ax.errorbar(phi_mids, phi_flat, xerr=xerr, marker="o", markersize=6, label="Unfolded from simulated light output")
# if sim_type == SimType.AMBE:
#     ax.plot(ambe_E, ambe_N * ambe_scaling, marker="o", markersize=6, label="AmBe ISO standard")
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
plt.show()

### End

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

### Animations

#### Input

In [ ]:
print(f"You have {len(errors)} frames of phi to animate.")
showevery_phi = helpers.get_input_with_default("Enter n for 'Show every n'th frame', or press Enter for default (1) >",
                                           1, int)

In [ ]:
print(f"You have {len(errors)} frames of weights to animate.")
showevery_weight = helpers.get_input_with_default("Enter n for 'Show every n'th frame', or press Enter for default (1) >",
                                           1, int)

#### Generate Animations

##### Phi

In [ ]:
phis = unfold_info["phis"]
phis_flat = [phi.counts.reshape(-1) for phi in phis]

In [ ]:
import matplotlib.animation as animation

phis_to_show = phis_flat[showevery_phi-1::showevery_phi]
plot_max = np.nanmax(phis_to_show[-1])

with plt.ioff():
    fig, ax = plt.subplots(figsize=figsize)
    phi_plot = ax.plot(phi_mids, phis_to_show[0], marker="o", markersize=6)[0]
    plot_frame = ax.annotate(f"Frame {showevery_phi}", (1, 1), xycoords="axes fraction", horizontalalignment="right", verticalalignment="top")
    if sim_type == 2:
        ax.plot(ambe_E, ambe_N * ambe_scaling, marker="o", markersize=6)
    # ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
    # if len(peaks) > 0:
    #     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
    #     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
    #         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
    ax.set(
        xlabel="E (MeV)",
        ylabel="Counts",
        # ylim=(0, 0.05),
        ylim=(0, plot_max * 1.1),
        title="Unfolded Spectrum (Simulated AmBe Neutrons)")
    
    def update(frame):
        y = phis_to_show[frame]
        phi_plot.set_ydata(y)
        plot_frame.set_text(f"Frame {(frame + 1)*showevery_phi}")
        return (phi_plot,plot_frame)
    
    ani = animation.FuncAnimation(fig=fig, func=update, frames=len(phis_to_show), interval=120)
    ani.save("ambe_phis.gif")

##### Phi Complete

In [ ]:
print("Phi Animation Complete")

##### Weights

In [ ]:
weights = unfold_info["weights"]
W_L_mids, W_phi_mids = weights[0].midpoints
print(W_L_mids.shape)
print(W_phi_mids.shape)
print(weights[0].shape)

L_widths = (W_L_mids[1:] - W_L_mids[:-1])
L_widths = np.insert(L_widths, (0,), L_widths[0])
phi_widths = W_phi_mids[1:] - W_phi_mids[:-1]
phi_widths = np.insert(phi_widths, (0,), phi_widths[0])

_x = W_phi_mids - (phi_widths / 2)
_y = W_L_mids - (L_widths / 2)
_xx, _yy = np.meshgrid(_x, _y)
x, y = np.ravel(_xx), np.ravel(_yy)

_xxw, _yyw = np.meshgrid(phi_widths, L_widths)
xw, yw = np.ravel(_xxw), np.ravel(_yyw)

tops = [np.ravel(weight.counts) for weight in weights]
bottom = np.zeros_like(tops[0])

Zmax_list = [np.nanmax(top) for top in tops]
Zmax = max(Zmax_list)
Zmin = 0

In [ ]:
import matplotlib as mpl

vaporwave_colors = [
    [255, 255, 255],
    [128, 69, 229],
    [75,127,255],
    [0,255,255],
    [255,186,129],
    [255,209,86],
    [252,120,183]
]
vaporwave_colors = [[value/255 for value in color] for color in vaporwave_colors]
vaporwave = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", vaporwave_colors, N=256)
# vaporwave = vaporwave.resampled(256)
vaporwave

In [ ]:
norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
sc.set_array([])

weights_colors = [vaporwave(norm(top)) for top in tops]

In [ ]:
# fig, ax = plt.subplots(figsize=figsize, projection="3d")
with plt.ioff():
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection="3d")
    
    artists = []
    _frame = 0
    for i in range(showevery_weight-1, len(tops), showevery_weight):
        print(f"{_frame % 10}", end="")
        _frame += 1
        top = tops[i]
        color = weights_colors[i]
        nan_mask = ~np.isnan(top)
        masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
        masked_color = color[nan_mask]
        # W_plot = ax.bar3d(x, y, bottom, xw, yw, top, color=color)
        W_plot = ax.bar3d(*masked_args, color=masked_color)
        frame = ax.annotate(f"Frame {i+1}", (1, 1), xycoords="axes fraction", horizontalalignment="right", verticalalignment="top")
        artists.append([W_plot, frame])

In [ ]:
# print([type(x) for x in artists])
ax.set(xlabel="Neutron energy (MeV)", ylabel="Light output (MeVee)", zlabel="Weight")
ani = animation.ArtistAnimation(fig=fig, artists=artists, interval=120)
ani.save("ambe_weights.gif")
# plt.show()

##### Weights Complete

In [ ]:
print("Weights Animation Complete")

## End

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()